In [ ]:
from google.colab import drive
import pandas as pd
import glob

# Google Drive mount karein (Connect popup aayega to Allow karein)
drive.mount('/content/drive')

# Truecaller data folder ki pehli file inspect karein
files = glob.glob('/content/drive/MyDrive/Truecaller data/*.csv')
print(f"Total CSV files found: {len(files)}")

if files:
    df_sample = pd.read_csv(files[0], nrows=5)
    print("\nColumns found in CSV:")
    print(df_sample.columns.tolist())
    print("\nFirst 3 rows:")
    display(df_sample.head(3))


Mounted at /content/drive
Total CSV files found: 29

Columns found in CSV:
['Number', 'Carrier', 'Name', 'Gender', 'Image', 'Address', 'JobTitle', 'CompanyName', 'Email', 'Website', 'Facebook', 'Twitter', 'Tags', 'Badges', 'Score', 'SpamCount']

First 3 rows:


,Number,Carrier,Name,Gender,Image,Address,JobTitle,CompanyName,Email,Website,Facebook,Twitter,Tags,Badges,Score,SpamCount
0,917044000354,Airtel,Sanjay 2,NaN,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.301211,0
1,917044000458,Airtel,Aman,NaN,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.317719,0
2,917044000999,Airtel,Priyanka Roy,NaN,NaN,Kolkata,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.347302,0


In [ ]:
import glob
import re
import json
import pandas as pd

files = glob.glob('/content/drive/MyDrive/Truecaller data/*.csv')
print(f"Processing {len(files)} files...")

male_names = set()
female_names = set()
total_gender_rows = 0

for idx, f in enumerate(files, 1):
    try:
        # Sirf Name aur Gender read karein taaki speed super fast rahe
        df = pd.read_csv(f, usecols=['Name', 'Gender'], low_memory=False)
        valid = df.dropna(subset=['Name', 'Gender'])

        for _, row in valid.iterrows():
            g = str(row['Gender']).strip().lower()
            if g in ['nan', '', 'unknown', 'null', 'none']:
                continue

            raw_nm = str(row['Name']).strip().lower()
            # Clean first name token
            tokens = [re.sub(r'[^a-z]', '', t) for t in raw_nm.split() if t]
            if not tokens:
                continue
            token = tokens[0]

            if len(token) >= 2:
                if g in ['m', 'male', 'boy', '1']:
                    male_names.add(token)
                    total_gender_rows += 1
                elif g in ['f', 'female', 'girl', '2']:
                    female_names.add(token)
                    total_gender_rows += 1

        print(f"[{idx}/{len(files)}] Done: {f.split('/')[-1]} | Total valid rows so far: {total_gender_rows}")
    except Exception as e:
        print(f"Error in {f.split('/')[-1]}: {e}")

# Overlapping conflicts remove karein
conflicts = male_names.intersection(female_names)
male_names -= conflicts
female_names -= conflicts

print("\n" + "="*40)
print(f"Total Rows with Gender Found: {total_gender_rows}")
print(f"Unique Male First Names: {len(male_names)}")
print(f"Unique Female First Names: {len(female_names)}")

# JSON file save karein Drive par
out_data = {
    "male": sorted(list(male_names)),
    "female": sorted(list(female_names))
}

out_path = '/content/drive/MyDrive/Truecaller_names_db.json'
with open(out_path, 'w', encoding='utf-8') as fp:
    json.dump(out_data, fp, ensure_ascii=False)

print(f"Saved successfully to: {out_path}")

Processing 29 files...
[1/29] Done: 917044000000_TrueCaller_(164759).csv | Total valid rows so far: 5120
[2/29] Done: 918420000000_TrueCaller_(193707).csv | Total valid rows so far: 11827
[3/29] Done: 918582000000_TrueCaller_(53531).csv | Total valid rows so far: 13348
[4/29] Done: 918583000000_TrueCaller_(58147).csv | Total valid rows so far: 15217
[5/29] Done: 918584000000_TrueCaller_(46974).csv | Total valid rows so far: 16808
[6/29] Done: 919007000000_TrueCaller_(226169).csv | Total valid rows so far: 26199
[7/29] Done: 919163000000_TrueCaller_(208258).csv | Total valid rows so far: 33913
[8/29] Done: 919748000000_TrueCaller_(204160).csv | Total valid rows so far: 42110
[9/29] Done: 919831000000_TrueCaller_(311707).csv | Total valid rows so far: 58712
[10/29] Done: 919903000000_TrueCaller_(233530).csv | Total valid rows so far: 68598
[11/29] Done: 917439000000_TrueCaller_(7352).csv | Total valid rows so far: 68839
[12/29] Done: 918981000000_TrueCaller_(126416).csv | Total valid row

In [ ]:
import json
import os

# 1. Truecaller extracted data load karein
with open('/content/drive/MyDrive/Truecaller_names_db.json', 'r', encoding='utf-8') as f:
    tc_data = json.load(f)

male_set = set(tc_data.get('male', []))
female_set = set(tc_data.get('female', []))

# 2. Hamare core high-priority overrides (Zainab, Raju, Bharat, etc. secure rakhne ke liye)
core_female = {
    "zainab", "zaynab", "putul", "mohar", "sath", "sathi", "tithe", "chhaya", "dulu", "kiran",
    "shaheen", "poonam", "vrinda", "naheed", "sudipta", "papiya", "tabinda", "june",
    "ritu", "radha", "saranya", "rupal", "rikhiya", "tuku", "chandra", "siya",
    "swagata", "mumtaz", "mehnaz", "pratibha", "venus", "taniya", "manmun", "raziya", "sultana",
    "tusi", "sharmista", "shrestha", "rina", "shabnam", "poli", "auswa", "antara"
}

core_male = {
    "sudhansu", "raju", "bablu", "bismu", "bishu", "desbandhu", "desbondhu", "laltu", "nuru",
    "pinku", "mintu", "titu", "pintu", "dukha", "sarabindu", "somu", "shanu", "suvendu",
    "pappu", "sonu", "santanu", "arnab", "mantu", "anjisnu", "batul", "priyansu", "nintu",
    "sirsendu", "diglu", "saju", "nihar", "sukhendu", "gullu", "soumendu", "dibyatanu",
    "abhimanyu", "ribhu", "riju", "ratul", "nehru", "puspendu", "roki", "toni", "bisnu",
    "gauranga", "surja", "mani", "sabasachi", "sabyasachi", "parsanta", "saugata",
    "hari", "rajarshi", "sunny", "banty", "bharat", "susanta", "arkaprava",
    "bubai", "roni", "naveen", "anthony", "bapi", "sushanta", "kanhaiya", "raja",
    "chowdhury", "praveen", "prashant", "prashanta", "ali", "imran", "ilyas"
}

# Conflicts clean karein
conflicts = male_set.intersection(female_set)
male_set -= conflicts
female_set -= conflicts

# Core overrides strictly inject karein
male_set.update(core_male)
female_set.update(core_female)
female_set -= core_male
male_set -= core_female

print(f"Final Merged Male Names: {len(male_set)}")
print(f"Final Merged Female Names: {len(female_set)}")
print(f"Total Master Corpus: {len(male_set) + len(female_set)}")

# Final names_db.json save karein
master_db = {
    "male": sorted(list(male_set)),
    "female": sorted(list(female_set))
}

out_file = '/content/drive/MyDrive/names_db.json'
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(master_db, f, ensure_ascii=False)

print(f"\n Master file ready at: {out_file}")

Final Merged Male Names: 14197
Final Merged Female Names: 5065
Total Master Corpus: 19262

 Master file ready at: /content/drive/MyDrive/names_db.json


In [2]:
from google.colab import drive
import os
import json
import joblib
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Drive Mount karein (Popup aane par 'Connect to Google Drive' allow karein)
drive.mount('/content/drive')

# 2. File path automatically dhoondein (names_db.json ya Truecaller_names_db.json)
possible_paths = [
    '/content/drive/MyDrive/names_db.json',
    '/content/drive/MyDrive/Truecaller_names_db.json',
    '/content/drive/MyDrive/Truecaller data/names_db.json'
]

json_file = None
for p in possible_paths:
    if os.path.exists(p):
        json_file = p
        break

if not json_file:
    raise FileNotFoundError("Dono files nahi mili. Kripya check karein ki file Drive par kis naam se hai.")

print(f"Loading data from: {json_file}")

with open(json_file, 'r', encoding='utf-8') as f:
    db = json.load(f)

print(f"Loaded {len(db.get('male', []))} Male names and {len(db.get('female', []))} Female names.")

# 3. Training data prepare karein
names = db['male'] + db['female']
labels = ['Male'] * len(db['male']) + ['Female'] * len(db['female'])

# 4. Train / Test Split
X_train, X_test, y_train, y_test = train_test_split(names, labels, test_size=0.1, random_state=42)

# 5. Character N-Gram Pipeline
model_pipeline = Pipeline([
    ('vect', CountVectorizer(analyzer='char_wb', ngram_range=(2, 4))),
    ('tfidf', TfidfTransformer()),
    ('clf', LogisticRegression(max_iter=1000, C=2.0))
])

# 6. Train karein
print("Model training chalu ho gayi...")
model_pipeline.fit(X_train, y_train)

# 7. Accuracy report print karein
y_pred = model_pipeline.predict(X_test)
print("\n--- Accuracy Report ---")
print(classification_report(y_test, y_pred))

# 8. Model save karein Drive par
save_path = '/content/drive/MyDrive/gender_model.pkl'
joblib.dump(model_pipeline, save_path)
print(f"\nModel safalta se save ho gaya: {save_path}")

Mounted at /content/drive
Loading data from: /content/drive/MyDrive/names_db.json
Loaded 14197 Male names and 5065 Female names.
Model training chalu ho gayi...

--- Accuracy Report ---
              precision    recall  f1-score   support

      Female       0.81      0.66      0.73       493
        Male       0.89      0.95      0.92      1434

    accuracy                           0.87      1927
   macro avg       0.85      0.80      0.82      1927
weighted avg       0.87      0.87      0.87      1927


Model safalta se save ho gaya: /content/drive/MyDrive/gender_model.pkl
